# Question 1 [1 mark]
Load train.csv into a Pandas DataFrame. Select only the 6 columns: GrLivArea, YearBuilt, LotArea,
OverallQual, Neighborhood, and SalePrice. Then:
1. Print the shape of the dataset and show the first 5 rows.

In [41]:
import warnings
warnings.filterwarnings('ignore')

In [42]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [43]:
from ydata_profiling import ProfileReport

In [44]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

In [68]:
from sklearn.linear_model import SGDRegressor, LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split

In [45]:
df = pd.read_csv("F:\Jupyter Projects\Phitron AL ML\Dataset\housing_train.csv")
df.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [46]:
selected_columns = ['GrLivArea', 'YearBuilt', 'LotArea', 'OverallQual', 'Neighborhood', 'SalePrice']
df = df[selected_columns]
display(df.head())
display(df.shape)

,GrLivArea,YearBuilt,LotArea,OverallQual,Neighborhood,SalePrice
0,1710,2003,8450,7,CollgCr,208500
1,1262,1976,9600,6,Veenker,181500
2,1786,2001,11250,7,CollgCr,223500
3,1717,1915,9550,7,Crawfor,140000
4,2198,2000,14260,8,NoRidge,250000


(1460, 6)

2. Check for missing values in each of the 6 columns. If any exist, fill numerical columns with their
median and categorical columns with their mode. Show the missing-value count before and after.

In [47]:
df.isnull().sum()

GrLivArea       0
YearBuilt       0
LotArea         0
OverallQual     0
Neighborhood    0
SalePrice       0
dtype: int64

In [48]:
numerical_col = df.select_dtypes(include = ['int64','float64']).columns.drop('SalePrice').to_list()
categorical_col = df.select_dtypes(include = ['object']).columns.to_list()


missing_val = Pipeline(steps = [
    ('imputer_val', SimpleImputer(strategy = 'median'))
])

missing_cat = Pipeline(steps = [
    ('imputer_cat', SimpleImputer(strategy = 'most_frequent'))
])

handle_missing_value = ColumnTransformer(
    transformers = [
        ('missing_val', missing_val, numerical_col),
        ('missing_cat', missing_cat, categorical_col)
    ],
    remainder = 'passthrough'
)

handle_missing_value.set_output(transform = 'pandas')

df = handle_missing_value.fit_transform(df)
df

,missing_val__GrLivArea,missing_val__YearBuilt,missing_val__LotArea,missing_val__OverallQual,missing_cat__Neighborhood,remainder__SalePrice
0,1710.0,2003.0,8450.0,7.0,CollgCr,208500
1,1262.0,1976.0,9600.0,6.0,Veenker,181500
2,1786.0,2001.0,11250.0,7.0,CollgCr,223500
3,1717.0,1915.0,9550.0,7.0,Crawfor,140000
4,2198.0,2000.0,14260.0,8.0,NoRidge,250000
...,...,...,...,...,...,...
1455,1647.0,1999.0,7917.0,6.0,Gilbert,175000
1456,2073.0,1978.0,13175.0,6.0,NWAmes,210000
1457,2340.0,1941.0,9042.0,7.0,Crawfor,266500
1458,1078.0,1950.0,9717.0,5.0,NAmes,142125


3. Print the descriptive statistics (mean, min, max, std) for GrLivArea, YearBuilt, and LotArea.
What do the large max values suggest about the need for preprocessing?

In [49]:
des_stat = ['missing_val__GrLivArea','missing_val__YearBuilt','missing_val__LotArea']

df[des_stat].describe()

,missing_val__GrLivArea,missing_val__YearBuilt,missing_val__LotArea
count,1460.000000,1460.000000,1460.000000
mean,1515.463699,1971.267808,10516.828082
std,525.480383,30.202904,9981.264932
min,334.000000,1872.000000,1300.000000
25%,1129.500000,1954.000000,7553.500000
50%,1464.000000,1973.000000,9478.500000
75%,1776.750000,2000.000000,11601.500000
max,5642.000000,2010.000000,215245.000000


At GrLivArea and LotArea's maximum value is far apart from 75% values. It seems like outliers. If we process with this data our model can't perform good for unseen data or also biased with this data. For this we need scaling.

In [50]:
profile = ProfileReport(df, title = 'Full Statistics for this housing dataset')
profile.to_file('stat.html')

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 6/6 [00:00<?, ?it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

# Question 2 [1 mark]
Explain in your own words (3–5 sentences) why Feature Scaling is mandatory before using
SGDRegressor. Then write code to apply StandardScaler only on GrLivArea, YearBuilt, and
LotArea. Print the mean and standard deviation of each column before and after scaling to verify
the transformation.

In [51]:
col = ['missing_val__GrLivArea','missing_val__YearBuilt','missing_val__LotArea']

display(df[col].mean())
display(df[col].std())

scaling_col = Pipeline(steps = [
    ('scaler', StandardScaler())
])

scaling_trans = ColumnTransformer(
    transformers = [
        ('num', scaling_col, col)
    ],
    remainder = 'passthrough'
)

scaling_trans.set_output(transform = 'pandas')

df = scaling_trans.fit_transform(df)
display(df.head(3))

col = ['num__missing_val__GrLivArea','num__missing_val__YearBuilt','num__missing_val__LotArea']
display(df[col].mean())
display(df[col].std())

missing_val__GrLivArea     1515.463699
missing_val__YearBuilt     1971.267808
missing_val__LotArea      10516.828082
dtype: float64

missing_val__GrLivArea     525.480383
missing_val__YearBuilt      30.202904
missing_val__LotArea      9981.264932
dtype: float64

,num__missing_val__GrLivArea,num__missing_val__YearBuilt,num__missing_val__LotArea,remainder__missing_val__OverallQual,remainder__missing_cat__Neighborhood,remainder__remainder__SalePrice
0,0.370333,1.050994,-0.207142,7.0,CollgCr,208500
1,-0.482512,0.156734,-0.091886,6.0,Veenker,181500
2,0.515013,0.984752,0.073480,7.0,CollgCr,223500


num__missing_val__GrLivArea   -1.277517e-16
num__missing_val__YearBuilt    1.046347e-15
num__missing_val__LotArea     -5.840077e-17
dtype: float64

num__missing_val__GrLivArea    1.000343
num__missing_val__YearBuilt    1.000343
num__missing_val__LotArea      1.000343
dtype: float64

In [52]:
df.columns

Index(['num__missing_val__GrLivArea', 'num__missing_val__YearBuilt',
       'num__missing_val__LotArea', 'remainder__missing_val__OverallQual',
       'remainder__missing_cat__Neighborhood',
       'remainder__remainder__SalePrice'],
      dtype='object')

In [53]:
df = df.rename(columns = {
    'num__missing_val__GrLivArea' : 'GrLivArea',
    'num__missing_val__YearBuilt' : 'YearBuilt',
    'num__missing_val__LotArea' : 'LotArea',
    'remainder__missing_val__OverallQual' : 'OverallQual',
    'remainder__missing_cat__Neighborhood' : 'Neighborhood',
    'remainder__remainder__SalePrice' : 'SalePrice'
}
)

In [54]:
df

,GrLivArea,YearBuilt,LotArea,OverallQual,Neighborhood,SalePrice
0,0.370333,1.050994,-0.207142,7.0,CollgCr,208500
1,-0.482512,0.156734,-0.091886,6.0,Veenker,181500
2,0.515013,0.984752,0.073480,7.0,CollgCr,223500
3,0.383659,-1.863632,-0.096897,7.0,Crawfor,140000
4,1.299326,0.951632,0.375148,8.0,NoRidge,250000
...,...,...,...,...,...,...
1455,0.250402,0.918511,-0.260560,6.0,Gilbert,175000
1456,1.061367,0.222975,0.266407,6.0,NWAmes,210000
1457,1.569647,-1.002492,-0.147810,7.0,Crawfor,266500
1458,-0.832788,-0.704406,-0.080160,5.0,NAmes,142125


# Question 3 [1 mark]
The dataset contains two categorical-type columns:
1. OverallQual (1–10 ordinal rating): Even though it is stored as a number, explain in 2–3
sentences why it should be treated as a categorical feature and encoded with OneHotEncoder
rather than passed as a raw number.

In [55]:
display(df['OverallQual'].value_counts())

ohe = OneHotEncoder(sparse_output = False).set_output(transform='pandas')

ohe_df = ohe.fit_transform(df[['OverallQual']])
df = pd.concat([df.drop(columns=['OverallQual']), ohe_df], axis = 1)
df.head(3)


OverallQual
5.0     397
6.0     374
7.0     319
8.0     168
4.0     116
9.0      43
3.0      20
10.0     18
2.0       3
1.0       2
Name: count, dtype: int64

,GrLivArea,YearBuilt,LotArea,Neighborhood,SalePrice,OverallQual_1.0,OverallQual_2.0,OverallQual_3.0,OverallQual_4.0,OverallQual_5.0,OverallQual_6.0,OverallQual_7.0,OverallQual_8.0,OverallQual_9.0,OverallQual_10.0
0,0.370333,1.050994,-0.207142,CollgCr,208500,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
1,-0.482512,0.156734,-0.091886,Veenker,181500,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
2,0.515013,0.984752,0.073480,CollgCr,223500,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0


2. Neighborhood (nominal text): Apply OneHotEncoder(sparse_output=False) to both OverallQual
and Neighborhood. Print the shape of the encoded array and list the first 3 new column names
generated.

In [56]:
display(df['Neighborhood'].value_counts())

ohe = OneHotEncoder(sparse_output = False).set_output(transform='pandas')

ohe_df = ohe.fit_transform(df[['Neighborhood']])
df = pd.concat([df.drop(columns=['Neighborhood']), ohe_df], axis = 1)
df.head(3)


Neighborhood
NAmes      225
CollgCr    150
OldTown    113
Edwards    100
Somerst     86
Gilbert     79
NridgHt     77
Sawyer      74
NWAmes      73
SawyerW     59
BrkSide     58
Crawfor     51
Mitchel     49
NoRidge     41
Timber      38
IDOTRR      37
ClearCr     28
StoneBr     25
SWISU       25
MeadowV     17
Blmngtn     17
BrDale      16
Veenker     11
NPkVill      9
Blueste      2
Name: count, dtype: int64

,GrLivArea,YearBuilt,LotArea,SalePrice,OverallQual_1.0,OverallQual_2.0,OverallQual_3.0,OverallQual_4.0,OverallQual_5.0,OverallQual_6.0,...,Neighborhood_NoRidge,Neighborhood_NridgHt,Neighborhood_OldTown,Neighborhood_SWISU,Neighborhood_Sawyer,Neighborhood_SawyerW,Neighborhood_Somerst,Neighborhood_StoneBr,Neighborhood_Timber,Neighborhood_Veenker
0,0.370333,1.050994,-0.207142,208500,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,-0.482512,0.156734,-0.091886,181500,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,0.515013,0.984752,0.073480,223500,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [69]:
X = df.drop(columns=['SalePrice'])
y = df['SalePrice']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [70]:
ohe_cols = [col for col in X.columns if col not in ['GrLivArea', 'YearBuilt', 'LotArea']]
num_cols = ['GrLivArea', 'YearBuilt', 'LotArea']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), ohe_cols)
    ],
    remainder='drop'
)

sgd_pipe = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', SGDRegressor(max_iter=1000, random_state=42))
])

In [71]:
print(X_train.index[:10].tolist())
print(X_test.index[:10].tolist())

[254, 1066, 638, 799, 380, 303, 86, 1385, 265, 793]
[892, 1105, 413, 522, 1036, 614, 218, 1160, 649, 887]


In [72]:
try:
    X_train[0]
except KeyError as e:
    print(f"KeyError: {e}")

print(X_train.iloc[0])

KeyError: 0
GrLivArea              -0.383521
YearBuilt              -0.472560
LotArea                -0.212153
OverallQual_1.0         0.000000
OverallQual_2.0         0.000000
OverallQual_3.0         0.000000
OverallQual_4.0         0.000000
OverallQual_5.0         1.000000
OverallQual_6.0         0.000000
OverallQual_7.0         0.000000
OverallQual_8.0         0.000000
OverallQual_9.0         0.000000
OverallQual_10.0        0.000000
Neighborhood_Blmngtn    0.000000
Neighborhood_Blueste    0.000000
Neighborhood_BrDale     0.000000
Neighborhood_BrkSide    0.000000
Neighborhood_ClearCr    0.000000
Neighborhood_CollgCr    0.000000
Neighborhood_Crawfor    0.000000
Neighborhood_Edwards    0.000000
Neighborhood_Gilbert    0.000000
Neighborhood_IDOTRR     0.000000
Neighborhood_MeadowV    0.000000
Neighborhood_Mitchel    0.000000
Neighborhood_NAmes      1.000000
Neighborhood_NPkVill    0.000000
Neighborhood_NWAmes     0.000000
Neighborhood_NoRidge    0.000000
Neighborhood_NridgHt    0.00000

In [73]:
sgd_pipe.fit(X_train, y_train)
print("Scaled SGD R²:", sgd_pipe.score(X_test, y_test))

Scaled SGD R²: 0.8245618500892878


In [74]:
explode_model = SGDRegressor(learning_rate='constant', eta0=0.01, max_iter=1, warm_start=True, random_state=42)

X_raw = X_train[['GrLivArea', 'YearBuilt', 'LotArea']].fillna(X_train[['GrLivArea', 'YearBuilt', 'LotArea']].median())

for i in range(1, 6):
    explode_model.fit(X_raw, y_train)
    loss = np.mean((explode_model.predict(X_raw) - y_train.values) ** 2)
    print(f"Iteration {i} | MSE Loss: {loss:.4e}")

Iteration 1 | MSE Loss: 2.5102e+09
Iteration 2 | MSE Loss: 2.5116e+09
Iteration 3 | MSE Loss: 2.5116e+09
Iteration 4 | MSE Loss: 2.5116e+09
Iteration 5 | MSE Loss: 2.5116e+09


In [75]:
X_raw_test = X_test[['GrLivArea', 'YearBuilt', 'LotArea']].fillna(X_train[['GrLivArea', 'YearBuilt', 'LotArea']].median())

print("Exploding SGD R²:", explode_model.score(X_raw_test, y_test))
print("Scaled SGD R²   :", sgd_pipe.score(X_test, y_test))

Exploding SGD R²: 0.6688127857446482
Scaled SGD R²   : 0.8245618500892878


In [76]:
try:
    sgd_pipe.coef_
except AttributeError as e:
    print(f"AttributeError: {e}")

AttributeError: 'Pipeline' object has no attribute 'coef_'


In [77]:
coef = sgd_pipe.named_steps['regressor'].coef_
intercept = sgd_pipe.named_steps['regressor'].intercept_

print("First 5 coef:", coef[:5])
print("Intercept:", intercept)
print("Total coef:", len(coef))

First 5 coef: [30359.83314458 19487.04041493  8734.66007247  9946.02280135
 -3716.72146749]
Intercept: [6258.16800633]
Total coef: 73


In [78]:
X_ne = df[['GrLivArea', 'LotArea']].values
y_ne = df['SalePrice'].values

X_mean, X_std = X_ne.mean(axis=0), X_ne.std(axis=0)
X_scaled = (X_ne - X_mean) / X_std

X_ne_train, X_ne_test, y_ne_train, y_ne_test = train_test_split(X_scaled, y_ne, test_size=0.2, random_state=42)

In [79]:
X_b_train = np.concatenate([np.ones((X_ne_train.shape[0], 1)), X_ne_train], axis=1)
X_b_test  = np.concatenate([np.ones((X_ne_test.shape[0],  1)), X_ne_test],  axis=1)

In [80]:
theta = np.dot(np.linalg.inv(np.dot(X_b_train.T, X_b_train)), np.dot(X_b_train.T, y_ne_train))
print("Theta:", theta)

Theta: [180150.64749652  52040.92714872   6248.67547327]


In [81]:
y_pred = np.dot(X_b_test, theta)

ss_res = np.sum((y_ne_test - y_pred) ** 2)
ss_tot = np.sum((y_ne_test - np.mean(y_ne_test)) ** 2)
r2 = 1 - (ss_res / ss_tot)

print("Normal Equation R²:", r2)

Normal Equation R²: 0.5580981658239974


In [82]:
from sklearn.linear_model import LinearRegression

lr_pipe = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

lr_pipe.fit(X_train, y_train)

print("LinearRegression R²:", lr_pipe.score(X_test, y_test))
print("SGDRegressor R²    :", sgd_pipe.score(X_test, y_test))

LinearRegression R²: 0.8428266252236922
SGDRegressor R²    : 0.8245618500892878


In [83]:
pd.DataFrame({
    'Model'  : ['LinearRegression', 'SGDRegressor'],
    'Method' : ['Normal Equation (Analytical)', 'Gradient Descent (Iterative)'],
    'R²'     : [round(lr_pipe.score(X_test, y_test), 4), round(sgd_pipe.score(X_test, y_test), 4)]
})

,Model,Method,R²
0,LinearRegression,Normal Equation (Analytical),0.8428
1,SGDRegressor,Gradient Descent (Iterative),0.8246
